# ⚡ File 2 Link — One-Click Heroku Deploy (Container Stack)

Deploy **FileToLink** to **your own** Heroku account straight from Colab. Uses the **container stack** (`heroku.yml` + `Dockerfile`), **not** buildpacks.

**How to use:** *File → Save a copy in Drive* (so your own copy is yours), then run the cells top to bottom (▶). Fill the form fields in cells 2 and 3.

| Step | What it does |
|---|---|
| 1 | Installs the Heroku CLI + safety helpers |
| 2 | Logs in with your Heroku **email + API key** |
| 3 | You fill app name, env vars (validated before anything is created) |
| 4 | Creates the app, sets env vars, builds & deploys, then **auto-wipes your secrets** |
| 5+ | Logs, restart, update env vars, delete app |

**Before you start**
- Heroku API key: *Account settings → API Key → Reveal*.
- **Basic dyno needs a verified payment method** on your Heroku account.
- Region tip: **eu** is usually faster for Telegram (DC in Amsterdam) and for India than **us**.

**Your secrets are safe**
- Heroku key, `API_HASH`, `BOT_TOKEN`, `DATABASE_URI` and extra tokens are typed into **hidden prompts** (or Colab 🔑 Secrets with the same names). They are never saved in this notebook.
- Everything you enter lives only in *your own* Colab session and *your own* Heroku app. The notebook author never sees it.
- After a successful deploy the notebook **automatically deletes** the login file, the API key, all tokens held in memory, the downloaded source, and the Heroku CLI cache. The same wipe runs when the Colab runtime ends and when you run cell 8.
- Form fields (email, app name, IDs) are saved inside *your copy* of the notebook. Before you share a copy: *Edit → Clear all outputs* and blank those fields.
- Never paste secrets into `EXTRA_ENV`. Use the prompts for anything secret.


In [ ]:
#@title 1️⃣ Install Heroku CLI & helpers
import subprocess, os, re, json, shlex, shutil, time, getpass, urllib.request, atexit

def run(cmd, show=True, cwd=None, env=None):
    """Run a command, stream output live, return (returncode, full_output)."""
    if isinstance(cmd, str):
        cmd = shlex.split(cmd)
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, cwd=cwd, env=env, bufsize=1)
    out = []
    for line in p.stdout:
        out.append(line)
        if show:
            print(line, end="")
    p.wait()
    return p.returncode, "".join(out)

if shutil.which("heroku") is None:
    rc, _ = run(["bash", "-c", "curl -fsSL https://cli-assets.heroku.com/install.sh | sh"], show=False)
    assert rc == 0, "Heroku CLI install failed"
rc, out = run(["heroku", "--version"], show=False)
print("✅", out.strip())

# ───────── safety helpers ─────────
_SECRETS = globals().get("_SECRETS", {})

def wipe_everything(verbose=True):
    """Delete every credential/secret from this Colab session."""
    try:
        os.remove(os.path.expanduser("~/.netrc"))
    except FileNotFoundError:
        pass
    os.environ.pop("HEROKU_API_KEY", None)
    _SECRETS.clear()
    for name in ("HEROKU_API_KEY", "API_HASH", "BOT_TOKEN", "DATABASE_URI", "MULTI_TOKENS", "ENV"):
        globals().pop(name, None)
    shutil.rmtree("/content/f2l_src", ignore_errors=True)
    shutil.rmtree(os.path.expanduser("~/.cache/heroku"), ignore_errors=True)
    if verbose:
        print("🧹 Secrets wiped from this Colab session (login file, API key, tokens, source).")

def ensure_login():
    """Cells 5-8 need a login. Secrets are auto-wiped after deploy, so ask to log in again."""
    rc, _ = run(["heroku", "auth:whoami"], show=False)
    if rc != 0:
        raise SystemExit("🔒 Not logged in (secrets were auto-cleared). Run cell 2 again, then re-run this cell.")

atexit.register(wipe_everything, False)   # runs when the runtime / kernel exits


In [ ]:
#@title 2️⃣ Login to Heroku (email + API key)
HEROKU_EMAIL = ""  #@param {type:"string"}

HEROKU_EMAIL = HEROKU_EMAIL.strip()
assert re.match(r"^[^@\s]+@[^@\s]+\.[^@\s]+$", HEROKU_EMAIL), "❌ Enter a valid Heroku email"

HEROKU_API_KEY = None
try:
    from google.colab import userdata          # Colab 🔑 Secrets
    HEROKU_API_KEY = userdata.get("HEROKU_API_KEY")
except Exception:
    pass
if not HEROKU_API_KEY:
    HEROKU_API_KEY = getpass.getpass("Heroku API key (hidden): ").strip()
assert HEROKU_API_KEY, "❌ API key required"

# netrc lets both the CLI and `git push` authenticate without printing the key
netrc = os.path.expanduser("~/.netrc")
with open(netrc, "w") as f:
    for host in ("api.heroku.com", "git.heroku.com"):
        f.write(f"machine {host}\n  login {HEROKU_EMAIL}\n  password {HEROKU_API_KEY}\n")
os.chmod(netrc, 0o600)
os.environ["HEROKU_API_KEY"] = HEROKU_API_KEY
os.environ["GIT_TERMINAL_PROMPT"] = "0"

rc, out = run(["heroku", "auth:whoami"], show=False)
if rc != 0:
    raise SystemExit("❌ Login failed — check email / API key.\n" + out)
print("✅ Logged in as:", out.strip())

In [ ]:
#@title 3️⃣ App settings & environment variables
# ───────── App ─────────
APP_NAME   = ""   #@param {type:"string"}
REGION     = "eu"             #@param ["eu", "us"]
DYNO_TYPE  = "basic"          #@param ["basic", "eco", "standard-1x"]
REPO_URL   = "https://github.com/GouthamSER/FileToLink"  #@param {type:"string"}
BRANCH     = "main"           #@param {type:"string"}
REUSE_IF_EXISTS = True        #@param {type:"boolean"}

# ───────── Required env vars (non-secret) ─────────
API_ID      = ""   #@param {type:"string"}
LOG_CHANNEL = ""   #@param {type:"string"}

# ───────── Optional env vars (non-secret, blank = skip) ─────────
DATABASE_NAME = "FileToLink"  #@param {type:"string"}
ADMINS        = ""            #@param {type:"string"}
FSUB_CHANNEL  = ""            #@param {type:"string"}
AUTO_RESTART  = "True"       #@param ["False", "True"]

# ───────── Anything else: one KEY=VALUE per line ─────────
EXTRA_ENV = """
# SHORTLINK=True
# SHORTLINK_URL=api.shareus.io
# SHORTLINK_API=xxxx
# ISGD=True
# PING_INTERVAL=1200
"""

# ---------- secrets: NEVER stored in the notebook ----------
# Typed into hidden prompts (or read from Colab 🔑 Secrets). #@param fields are
# saved inside the notebook file, so secrets must not live there.
def ask_secret(name, label, required=True):
    if name in _SECRETS:
        return _SECRETS[name]
    val = None
    try:
        from google.colab import userdata
        val = userdata.get(name)
    except Exception:
        pass
    if not val:
        val = getpass.getpass(f"{label} {'(required)' if required else '(optional, Enter to skip)'}: ")
    _SECRETS[name] = (val or "").strip()
    return _SECRETS[name]

API_HASH      = ask_secret("API_HASH",  "API_HASH (my.telegram.org)")
BOT_TOKEN     = ask_secret("BOT_TOKEN", "BOT_TOKEN (@BotFather)")
DATABASE_URI  = ask_secret("DATABASE_URI", "MongoDB DATABASE_URI", required=False)
MULTI_TOKENS  = ask_secret("MULTI_TOKENS", "Extra bot tokens, space separated", required=False)

# ---------- validation ----------
errs = []
if not re.fullmatch(r"[a-z][a-z0-9-]{2,29}", APP_NAME):
    errs.append("APP_NAME: 3-30 chars, lowercase letters/digits/dashes, must start with a letter")
if not API_ID.strip().isdigit():
    errs.append("API_ID must be digits (my.telegram.org)")
if not re.fullmatch(r"[0-9a-fA-F]{32}", API_HASH.strip()):
    errs.append("API_HASH must be 32 hex chars (my.telegram.org)")
if not re.fullmatch(r"\d{6,}:[\w-]{30,}", BOT_TOKEN.strip()):
    errs.append("BOT_TOKEN looks wrong (expected 123456:ABC... from @BotFather)")
if ADMINS.strip() and not re.fullmatch(r"-?\d+(\s+-?\d+)*", ADMINS.strip()):
    errs.append("ADMINS must be numeric user ids separated by spaces")
if not re.fullmatch(r"-?\d+", LOG_CHANNEL.strip()):
    errs.append("LOG_CHANNEL must be a numeric id like -1001234567890")
if FSUB_CHANNEL.strip() and not re.fullmatch(r"-?\d+", FSUB_CHANNEL.strip()):
    errs.append("FSUB_CHANNEL must be a numeric id")
if DATABASE_URI.strip() and not DATABASE_URI.strip().startswith(("mongodb://", "mongodb+srv://")):
    errs.append("DATABASE_URI must start with mongodb:// or mongodb+srv://")
if not REPO_URL.startswith("https://"):
    errs.append("REPO_URL must be an https:// git URL")
if errs:
    raise SystemExit("❌ Fix these first:\n - " + "\n - ".join(errs))

# ---------- build env dict ----------
ENV = {
    "API_ID": API_ID.strip(), "API_HASH": API_HASH.strip(),
    "BOT_TOKEN": BOT_TOKEN.strip(), "LOG_CHANNEL": LOG_CHANNEL.strip(),
    "DATABASE_NAME": DATABASE_NAME.strip() or "FileToLink",
    "AUTO_RESTART": AUTO_RESTART,
}
for k, v in (("DATABASE_URI", DATABASE_URI), ("ADMINS", ADMINS), ("FSUB_CHANNEL", FSUB_CHANNEL)):
    if v.strip():
        ENV[k] = v.strip()
for i, tok in enumerate(MULTI_TOKENS.split(), 1):
    ENV[f"MULTI_TOKEN{i}"] = tok
for line in EXTRA_ENV.splitlines():
    line = line.strip()
    if not line or line.startswith("#") or "=" not in line:
        continue
    k, v = line.split("=", 1)
    if k.strip() and v.strip():
        ENV[k.strip()] = v.strip()

SECRET_KEYS = ("TOKEN", "HASH", "URI", "API", "KEY")
print("✅ Settings OK. Will set these config vars (secrets hidden):")
for k, v in ENV.items():
    shown = "••••••" if any(s in k for s in SECRET_KEYS) else v
    print(f"   {k} = {shown}")
print(f"\nApp: {APP_NAME} | region: {REGION} | dyno: {DYNO_TYPE} | stack: container")

In [ ]:
#@title 4️⃣ 🚀 Create app, set stack + env vars, build & deploy
AUTO_WIPE_SECRETS = True  #@param {type:"boolean"}

def heroku(*args, show=True):
    return run(["heroku", *args], show=show)

# 1) create (or reuse) the app
rc, _ = heroku("apps:info", "-a", APP_NAME, show=False)
if rc == 0:
    assert REUSE_IF_EXISTS, f"❌ App '{APP_NAME}' already exists. Tick REUSE_IF_EXISTS or change APP_NAME."
    print(f"ℹ️ App '{APP_NAME}' exists — reusing it.")
else:
    rc, out = heroku("apps:create", APP_NAME, "--region", REGION, show=False)
    if rc != 0:
        raise SystemExit("❌ Could not create app (name taken? account limit?):\n" + out)
    print(f"✅ Created app: {APP_NAME}")

# 2) container stack (Dockerfile build via heroku.yml)
rc, out = heroku("stack:set", "container", "-a", APP_NAME, show=False)
assert rc == 0, "❌ stack:set failed:\n" + out
print("✅ Stack = container")

# 3) web URL -> URL env var (links are built from it)
rc, out = heroku("apps:info", "-a", APP_NAME, "--json", show=False)
WEB_URL = json.loads(out)["app"]["web_url"]
ENV["URL"] = WEB_URL

# 4) config vars (output captured, never printed: CLI echoes values)
rc, out = heroku("config:set", "-a", APP_NAME, *[f"{k}={v}" for k, v in ENV.items()], show=False)
assert rc == 0, "❌ config:set failed:\n" + re.sub(r"=\S+", "=***", out)
print(f"✅ {len(ENV)} config vars set (URL = {WEB_URL})")

# 5) fetch source as a fresh git repo (Heroku rejects shallow pushes)
SRC = "/content/f2l_src"
shutil.rmtree(SRC, ignore_errors=True)
rc, out = run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, SRC], show=False)
assert rc == 0, "❌ git clone failed (private repo? wrong branch?):\n" + out
shutil.rmtree(os.path.join(SRC, ".git"), ignore_errors=True)
assert os.path.exists(os.path.join(SRC, "Dockerfile")), "❌ Dockerfile missing in repo"
yml = os.path.join(SRC, "heroku.yml")
if not os.path.exists(yml):
    open(yml, "w").write("build:\n  docker:\n    web: Dockerfile\n")
    print("ℹ️ heroku.yml was missing — created it")

run(["git", "init", "-q", "-b", "main"], cwd=SRC, show=False)
run(["git", "-c", "user.name=colab", "-c", "user.email=colab@localhost", "add", "-A"], cwd=SRC, show=False)
run(["git", "-c", "user.name=colab", "-c", "user.email=colab@localhost", "commit", "-qm", "deploy from colab"], cwd=SRC, show=False)

# 6) push -> Heroku builds the Docker image
print("\n🔨 Building image on Heroku (2-5 min)...\n")
rc, out = run(["git", "push", "--force", f"https://git.heroku.com/{APP_NAME}.git", "HEAD:main"], cwd=SRC)
assert rc == 0, "❌ Build/push failed — read the log above."

# 7) dyno type + scale
rc, out = heroku("ps:type", f"web={DYNO_TYPE}", "-a", APP_NAME, show=False)
if rc != 0:
    print(f"⚠️ Could not set dyno '{DYNO_TYPE}' (add a payment method at heroku.com/verify):\n{out.strip()}")
heroku("ps:scale", "web=1", "-a", APP_NAME, show=False)

# 8) health check
print("\n⏳ Waiting for the bot to boot...")
ok = False
for _ in range(20):
    time.sleep(6)
    try:
        r = urllib.request.urlopen(WEB_URL + "?json=true", timeout=10)
        print("🟢 Online:", r.read().decode()[:200]); ok = True; break
    except Exception:
        pass
print("\n🎉 DEPLOYED:", WEB_URL if ok else f"{WEB_URL}  (not answering yet — run cell 5 for logs)")

# 9) remove every secret from this session (re-login via cell 2 for logs / restart)
if AUTO_WIPE_SECRETS and ok:
    wipe_everything()
    print("ℹ️ Need logs or restart later? Run cell 2 once more, then cells 5 / 7.")
elif AUTO_WIPE_SECRETS:
    print("ℹ️ Secrets kept because the bot is not online yet (so you can fix and retry). Run cell 8 when finished.")


In [ ]:
#@title 5️⃣ 📜 View logs & dyno status
LOG_LINES = 150  #@param {type:"integer"}
ensure_login()
run(["heroku", "ps", "-a", APP_NAME])
print()
run(["heroku", "logs", "-n", str(LOG_LINES), "-a", APP_NAME])
print("\n⚠️ Logs contain your users' IP addresses. Run Edit → Clear all outputs before sharing this notebook.")


In [ ]:
#@title 6️⃣ ♻️ Update env vars only (run cell 2, edit + run cell 3, then run this)
ensure_login()
if "ENV" not in globals():
    raise SystemExit("❌ Run cell 3 first (it asks for your secrets again).")
rc, out = run(["heroku", "config:set", "-a", APP_NAME, *[f"{k}={v}" for k, v in ENV.items()]], show=False)
print("✅ Updated & restarted" if rc == 0 else "❌ " + re.sub(r"=\S+", "=***", out))
wipe_everything()


In [ ]:
#@title 7️⃣ 🔁 Restart  /  🔄 Redeploy latest code
ACTION = "restart"  #@param ["restart", "redeploy latest code"]
ensure_login()
if ACTION == "restart":
    run(["heroku", "ps:restart", "-a", APP_NAME])
else:
    print("Re-run cell 4 (REUSE_IF_EXISTS must be ticked): it re-pulls the repo and rebuilds.")

In [ ]:
#@title 8️⃣ 🧹 Cleanup  (wipe secrets from Colab / optionally DELETE the app)
WIPE_SECRETS_NOW = True   #@param {type:"boolean"}
DELETE_APP       = False  #@param {type:"boolean"}
CONFIRM_APP_NAME = ""     #@param {type:"string"}

if DELETE_APP:
    ensure_login()
    assert CONFIRM_APP_NAME == APP_NAME, "❌ Type the exact app name in CONFIRM_APP_NAME to delete."
    run(["heroku", "apps:destroy", APP_NAME, "--confirm", APP_NAME])
if WIPE_SECRETS_NOW:
    wipe_everything()
